# TrafficGuard AI — YOLOv8 Training
Train a custom YOLOv8s model on 5,254 dashcam images with 23 object classes.

**Runtime:** Change to GPU → `Runtime > Change runtime type > T4 GPU`

In [ ]:
# Step 1: Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Step 2: Install ultralytics
!pip install ultralytics -q

## Upload Dataset

Upload `traffic_dataset.zip` (created by the zip command below) to Colab.

**On your Mac terminal, run:**
```bash
cd /Users/pratyushpadhy/ML-Predictor
zip -r traffic_dataset.zip images/ labels/ -x '*.DS_Store'
```
Then upload `traffic_dataset.zip` using the file browser on the left (📁 icon), or run the cell below to upload.

In [ ]:
# Step 3: Upload dataset zip
from google.colab import files
uploaded = files.upload()  # Select traffic_dataset.zip

In [ ]:
# Step 4: Unzip dataset
!unzip -q traffic_dataset.zip -d /content/traffic_data
!echo "Train images:" && ls /content/traffic_data/images/train/ | wc -l
!echo "Val images:" && ls /content/traffic_data/images/val/ | wc -l
!echo "Train labels:" && ls /content/traffic_data/labels/train/ | wc -l
!echo "Val labels:" && ls /content/traffic_data/labels/val/ | wc -l

In [ ]:
# Step 5: Create data.yaml
data_yaml = """
path: /content/traffic_data
train: images/train
val: images/val
nc: 23
names:
  0: person
  1: car
  2: truck
  3: bus
  4: motorcycle
  5: red light
  6: green light
  7: stop sign
  8: no entry
  9: no overtaking
  10: speed limit 20
  11: speed limit 30
  12: speed limit 40
  13: speed limit 50
  14: speed limit 60
  15: speed limit 70
  16: speed limit 80
  17: speed limit 100
  18: speed limit 120
  19: no left turn
  20: no right turn
  21: no stopping
  22: no u-turn
"""

with open('/content/data.yaml', 'w') as f:
    f.write(data_yaml)

print("data.yaml created!")

In [ ]:
# Step 6: Train YOLOv8s
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data='/content/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=15,
    project='/content/runs',
    name='traffic_violation',
    device=0,  # NVIDIA GPU
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    flipud=0.0,
    fliplr=0.5,
)

In [ ]:
# Step 7: Validate
metrics = model.val()
print(f"\nmAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

In [ ]:
# Step 8: Show training results
from IPython.display import Image, display
display(Image('/content/runs/traffic_violation/results.png', width=800))

In [ ]:
# Step 9: Download best weights
from google.colab import files
files.download('/content/runs/traffic_violation/weights/best.pt')
print("\nDownload best.pt and place it at: backend/model/best.pt")